In [0]:
list = [("Spring", 12.3),
("Summer", 10.5),
("Autumn", 8.2),
("Winter", 15.1)]

In [0]:
df = spark.createDataFrame(list).toDF("Season","Temp")

In [0]:
df.show()

In [0]:
lib_df = spark.read.json("dbfs:/FileStore/datasets/library_data.json")

In [0]:
lib_df.show(truncate=False)

In [0]:
{"library_name": "Central Library","location": "City Center","books": [{"book_id": "B001","book_name": "The Great Gatsby","author": "F. Scott Fitzgerald","copies_available": 5},{"book_id": "B002","book_name": "To Kill a Mockingbird","author": "Harper Lee","copies_available": 3}],"members": [{"member_id": "M001","member_name": "John Smith","age": 28,"books_borrowed": ["B001"]},{"member_id": "M002","member_name": "Emma Johnson","age": 35,"books_borrowed": []}]},
{"library_name": "Community Library","location": "Suburb","books": [{"book_id": "B003","book_name": "1984","author": "George Orwell","copies_available": 2},{"book_id": "B004","book_name": "Pride and Prejudice","author": "Jane Austen","copies_available": 4}],"members": [{"member_id": "M003","member_name": "Michael Brown","age": 42,"books_borrowed": ["B003","B004"]},{"member_id": "M004","member_name": "Sophia Davis","age": 31,"books_borrowed": ["B004"]}]}


In [0]:
from pyspark.sql.types import *

In [0]:
schema = StructType([
    StructField("library_name",StringType()),
    StructField("location",StringType()),
    StructField("books",StructType([
        StructField("book_id",IntegerType()),
        StructField("book_name",StringType()),
        StructField("author",StringType()),
        StructField("copies_available",IntegerType())
    ])),
    StructField("members",StructType([
        StructField("member_id",IntegerType()),
        StructField("member_name",StringType()),
        StructField("age",IntegerType()),
        StructField("books_borrowed",StructType([
            StructField("book_id",IntegerType())
        ]))
    ]))])


In [0]:
schema_struct = StructType([
StructField("library_name", StringType()),
StructField("location", StringType()),
StructField("books", ArrayType(
StructType([
StructField("book_id", StringType()),
StructField("book_name", StringType()),
StructField("author", StringType()),
StructField("copies_available", IntegerType())
])
)),
StructField("members", ArrayType(
StructType([
StructField("member_id", StringType()),
StructField("member_name", StringType()),
StructField("age", IntegerType()),
StructField("books_borrowed", ArrayType(StringType()))
])
))
])

In [0]:
lib_schema_df = spark.read.schema(schema_struct).json("dbfs:/FileStore/datasets/library_data.json")

In [0]:
lib_schema_df.show(truncate=False)

In [0]:
train_df = spark.read.csv("dbfs:/FileStore/datasets/train.csv",header=True,inferSchema = True)

In [0]:
train_df.show()

a) Drop the columns passenger_name and age from the dataset.
b) Count the number of rows after removing duplicates of columns
train_number and ticket_number.
c) Count the number of unique train names.

In [0]:
df = train_df.drop("passenger_name","age")

In [0]:
df.show()

In [0]:
df1 = train_df.dropDuplicates(["train_number","ticket_number"])

In [0]:
df1.count()

In [0]:
df2 = train_df.select("train_name").distinct()

In [0]:
df2.count()

In [0]:
schema = ("product string, quantity int, revenue double, store_id int")

In [0]:
sales_df = spark.read.schema(schema).json("dbfs:/FileStore/datasets/sales_data.json")

In [0]:
sales_df.show()

In [0]:
sales_df = spark.read.schema(schema).option("mode","permissive").json("dbfs:/FileStore/datasets/sales_data.json")

In [0]:
sales_df.show()

In [0]:
df = sales_df.count()
print(df)

In [0]:
sales_df_1 = spark.read.schema(schema).option("mode","dropMalformed").json("dbfs:/FileStore/datasets/sales_data.json")

In [0]:
sales_df_1.show()

In [0]:
sales_df_1.count()

In [0]:
sales_df_2 = spark.read.schema(schema).option("mode","failFast").json("dbfs:/FileStore/datasets/sales_data.json")

In [0]:
sales_df_2.show()

In [0]:
hos_df = spark.read.format("csv").option("header","true").option("inferSchema","true").load("dbfs:/FileStore/datasets/hospital.csv")

In [0]:
hos_df.show()

1. Drop the "doctor_id" column from the dataset.
2. Rename the "total_cost" column to "hospital_bill".
3. Add a new column called "duration_of_stay" that represents the number
of days a patient stayed in the hospital. (hint: The duration should be
calculated as the difference between the "discharge_date" and
"admission_date" columns.)
4. Create a new column called "adjusted_total_cost" that calculates the
adjusted total cost based on the diagnosis as follows:
If the diagnosis is "Heart Attack", multiply the hospital_bill by 1.5.
If the diagnosis is "Appendicitis", multiply the hospital_bill by 1.2.
For any other diagnosis, keep the hospital_bill as it is.
5. Select the "patient_id", "diagnosis", "hospital_bill", and
"adjusted_total_cost" columns.

In [0]:
doc_df = hos_df.drop("doctor_id").show()

In [0]:
bill_df = hos_df.withColumnRenamed("total_cost","hospital_bill").show()

In [0]:
from pyspark.sql.functions import expr

In [0]:
stay_df = hos_df.withColumn("duration_of_stay",expr("datediff(discharge_date,admission_date) as duration_of_stay")).show()

In [0]:
hos_df_1 = hos_df.withColumn("adjusted_total_cost",expr("CASE WHEN diagnosis like '%Heart Attack%' then total_cost * 1.5 WHEN diagnosis like '%Appendicitis%' then total_cost * 1.2 ELSE total_cost END "))

In [0]:
hos_df_1.show()

In [0]:
hos_df_1.withColumnRenamed("total_cost","hospital_bill").select("patient_id","diagnosis","adjusted_total_cost","hospital_bill").show()